# 문서 탐색 (EDA) — 제공 뉴스

벡터 DB 구성 전에 문서의 구성과 품질을 파악한다 (가이드 4-4).
청킹 방식, 전처리 규칙, 품질 필터(`indexer/quality.py`) 기준은 이 노트북에서 확인한 결과를 근거로 정하고 리포트 2장에 기술한다.

**실행 전**: 제공 CSV 를 `data/provided/` 에 넣고 `python -m collector provided` 로 적재한다.

> API 키나 키가 포함된 출력을 노트북에 남기지 않는다.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from IPython.display import display

from common.docstore import open_db
from indexer.quality import AMBIGUOUS_DISTRICTS, SEOUL_DISTRICTS

px.defaults.template = "plotly_white"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # 범주 색상 고정 순서 (1~3번)

with open_db(read_only=True) as conn:
    news = pd.read_sql_query("SELECT * FROM documents WHERE doc_type = 'news'", conn)

news["extra"] = news["extra"].map(json.loads)
news["published"] = pd.to_datetime(news["published_at"], errors="coerce")
news["text"] = news["raw_text"].fillna("")
news["body_len"] = news["text"].str.len()
news["file_group"] = news["extra"].map(
    lambda e: " + ".join(sorted({"소상공인" if f.startswith("01_") else "지역화폐" for f in e["provided_files"]}))
)
print(f"뉴스 {len(news):,}건")

## 1. 구성
- 파일(그룹)별, 수집 키워드별 건수
- 매체 분포와 매체명 결손 비율

In [ ]:
display(news["file_group"].value_counts().rename("건수").to_frame())

# 묶음 파일의 복수 키워드 표기 형식은 원본마다 다를 수 있어 흔한 구분자를 모두 나눈다.
keywords = news["keyword"].fillna("").str.split(r"\s*[;,|/]\s*", regex=True).explode()
display(keywords[keywords != ""].value_counts().rename("건수").to_frame())

missing_source = news["extra"].map(lambda e: e["source_missing"])
print(f"매체 수 {news['source'].nunique():,} / 매체명 결손 {missing_source.mean():.1%}")
display(news[missing_source].groupby(news["published"].dt.to_period("M")).size().rename("결손 건수").tail(12).to_frame())

top_sources = news["source"].value_counts().head(20).rename_axis("매체").reset_index(name="기사 수")
fig = px.bar(top_sources, x="기사 수", y="매체", orientation="h", title="매체별 기사 수 (상위 20)",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

## 2. 기간
- 발행일 분포, 특정 시기 편중 여부

In [ ]:
print("기간:", news["published"].min(), "~", news["published"].max())
print("발행일 미상:", news["published"].isna().sum())

monthly = (
    news.dropna(subset=["published"])
    .assign(month=lambda d: d["published"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "file_group"]).size().reset_index(name="기사 수")
)
fig = px.line(monthly, x="month", y="기사 수", color="file_group", title="월별 기사 수",
              color_discrete_sequence=SERIES, labels={"month": "", "file_group": "그룹"})
fig.update_traces(line_width=2)
fig.show()

## 3. 길이
- 본문 길이 분포, 과소·과다 문서 비율 → `QualityConfig.min_chars` 후보

In [ ]:
print(news["body_len"].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95, .99]).round(0))
for n in (100, 200, 300, 500):
    print(f"본문 {n}자 미만: {(news['body_len'] < n).mean():.1%}")

fig = px.histogram(news, x="body_len", nbins=100, title="본문 길이 분포 (상위 1% 제외)",
                   range_x=[0, news["body_len"].quantile(0.99)], labels={"body_len": "본문 길이(자)"},
                   color_discrete_sequence=SERIES[:1])
fig.update_layout(bargap=0.05, yaxis_title="기사 수")
fig.show()

## 4. 품질
- 사진·광고·인사 기사 표지, 저작권 등 보일러플레이트 비중

In [ ]:
TITLE_MARKERS = {
    "포토·사진·영상": r"^\s*[\[【(]\s*(?:포토|사진|화보|영상)",
    "광고·협찬": r"[\[【(]\s*(?:광고|AD|PR|협찬|후원)\s*[\]】)]",
    "인사·부고·동정": r"^\s*[\[【(]\s*(?:인사|부고|동정)",
}
BODY_MARKERS = {
    "저작권 문구": r"무단\s*전재|재배포\s*금지|Copyright|ⓒ|©",
    "AI 학습 금지 문구": r"AI\s*학습\s*(?:이용)?\s*금지",
    "기자 이메일": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "특수 줄바꿈 문자(LS·PS)": "[\u2028\u2029]",
}
rows = [("제목", k, news["title"].str.contains(p, regex=True).mean()) for k, p in TITLE_MARKERS.items()]
rows += [("본문", k, news["text"].str.contains(p, regex=True).mean()) for k, p in BODY_MARKERS.items()]
markers = pd.DataFrame(rows, columns=["위치", "표지", "비율"])
markers["비율"] = markers["비율"].map("{:.1%}".format)
display(markers)

for k, p in TITLE_MARKERS.items():
    print(f"--- {k} 예시")
    print(news.loc[news["title"].str.contains(p, regex=True), "title"].head(3).to_string())

## 5. 서울 비중 — 범위 필터 근거
- 통신사 기사의 첫머리 발신지 표기 `(서울=연합뉴스)` 는 기사 내용과 무관하므로 제거한 뒤 판정한다
- '서울'이 들어간 고유명사(서울경제, 서울신문, 서울대 등)와 중복 지명(중구·강서구)도 제외한다
- 키워드 단순 매칭이라 근사치다. 오판은 샘플을 읽어 확인한다

In [ ]:
LEAD_CHARS = 300  # 제목 + 본문 앞부분을 '리드'로 본다. QualityConfig.lead_chars 후보
DATELINE = r"[(\[]\s*[가-힣]{2,4}\s*=\s*[^)\]]{1,20}[)\]]"
unambiguous = [d for d in SEOUL_DISTRICTS if d not in AMBIGUOUS_DISTRICTS]
SEOUL = "서울(?!경제|신문|대학|대|우유|보증)|" + "|".join(unambiguous)

lead_raw = news["title"] + " " + news["text"].str.slice(0, LEAD_CHARS)
lead = lead_raw.str.replace(DATELINE, " ", regex=True)
news["seoul_lead"] = lead.str.contains(SEOUL, regex=True)
print(f"서울 언급 비율  발신지 표기 포함: {lead_raw.str.contains(SEOUL, regex=True).mean():.1%}"
      f"  →  제거 후: {news['seoul_lead'].mean():.1%}")
display(news.groupby("file_group")["seoul_lead"].mean().map("{:.1%}".format).rename("서울 언급 비율").to_frame())

# '경기'는 '경기 침체'와, '대전'은 '디지털 대전환'과 섞이므로 패턴을 좁힌다. '광주'는 경기 광주시와 섞인다.
REGIONS = {
    "서울": SEOUL, "부산": "부산", "대구": "대구", "인천": "인천", "광주": "광주",
    "대전": "대전(?!환)", "울산": "울산", "세종": "세종시|세종특별자치시", "경기": "경기도",
    "강원": "강원", "충북": "충북|충청북도", "충남": "충남|충청남도", "전북": "전북|전라북도",
    "전남": "전남|전라남도", "경북": "경북|경상북도", "경남": "경남|경상남도", "제주": "제주",
}
regions = pd.Series({r: lead.str.contains(p, regex=True).mean() for r, p in REGIONS.items()})
fig = px.bar(regions.rename_axis("지역").reset_index(name="비율"), x="비율", y="지역", orientation="h",
             title=f"제목·리드 {LEAD_CHARS}자 내 광역 지자체 언급 비율 (발신지 표기 제거 후)",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(bargap=0.35, height=560)
fig.show()

## 6. 중복 현황
- 같은 URL, 본문 완전 일치, 괄호 머리말을 뗀 제목 일치 → `indexer/dedup.py` 임계값 근거

In [ ]:
nonempty = news["text"] != ""
norm_title = (news["title"].str.replace(r"[\[【(][^\]】)]*[\]】)]", "", regex=True)
              .str.replace(r"\W+", "", regex=True))
dup = {
    "같은 URL (발행일만 다름)": news["url"].duplicated(keep=False).sum(),
    "본문 완전 일치": news.loc[nonempty, "text"].duplicated(keep=False).sum(),
    "정규화 제목 일치": norm_title[norm_title != ""].duplicated(keep=False).sum(),
}
display(pd.Series(dup, name="관련 문서 수").to_frame())

groups = news.assign(norm_title=norm_title).groupby("norm_title").filter(lambda g: len(g) > 1)
top = groups["norm_title"].value_counts().head(5).index
display(groups[groups["norm_title"].isin(top)][["norm_title", "published_at", "source", "title"]]
        .sort_values(["norm_title", "published_at"]))

## 7. 내용 — 샘플 직접 읽기
- 구조와 서술 방식, 제거할 반복 문구, 핵심 내용의 위치를 기록한다

In [ ]:
SAMPLE_PER_GROUP = 5
for group, g in news.groupby("file_group"):
    print(f"===== {group} =====")
    for _, r in g.sample(min(SAMPLE_PER_GROUP, len(g)), random_state=42).iterrows():
        print(f"[{r['published_at']}] {r['source']} | {r['title']}\n{r['text'][:300]}\n")

## 8. 부족 자료 탐색 — 제공되지 않은 가이드 키워드의 본문 등장 건수
- 등장이 적은 항목은 직접 수집(보도자료·서울시 자료·보고서·법령)으로 보강하거나 답변 불가 문항으로 쓴다

In [ ]:
GUIDE_TERMS = {
    "소상공인 정책자금": r"정책\s*자금",
    "상권활성화구역": r"상권\s*활성화\s*구역",
    "지역사랑상품권": r"지역사랑\s*상품권|서울사랑\s*상품권",
    "디지털 전환": r"디지털\s*대?전환",
    "상가 공실": r"공실",
    "상권 임대료": r"임대료",
    "젠트리피케이션": r"젠트리피케이션|둥지\s*내몰림",
    "생활인구": r"생활\s*인구",
    "카드·가맹점 수수료": r"(?:카드|가맹점)\s*수수료",
    "소상공인시장진흥공단": r"소상공인시장진흥공단|소진공",
}
seoul = news[news["seoul_lead"]]
coverage = pd.DataFrame({
    "전체": {k: news["text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
    "서울 언급 기사": {k: seoul["text"].str.contains(p, regex=True).sum() for k, p in GUIDE_TERMS.items()},
})
display(coverage)

fig = px.bar(coverage["서울 언급 기사"].rename_axis("항목").reset_index(name="기사 수"),
             x="기사 수", y="항목", orientation="h", title="서울 언급 기사 중 항목별 등장 건수",
             color_discrete_sequence=SERIES[:1])
fig.update_yaxes(categoryorder="total ascending", title=None)
fig.update_layout(bargap=0.35)
fig.show()

## 정리
- 전처리 규칙 (보일러플레이트·발신지 표기):
- 품질 필터 기준 후보 (`min_chars`, `lead_chars`, 주제어):
- 중복 처리 기준 후보:
- 청킹 전략 후보:
- 직접 수집 보강 대상 (8번 결과):